# Part 1 — Step 5: Detection Visualisation

Side-by-side comparison of predictions on test images:

```
Ground Truth  |  YOLOv8s  |  Faster R-CNN
```

**Output:** `outputs/figures/detection_comparison.png`

**Prerequisites:** Checkpoints from notebooks 02 and 03.

In [ ]:
# Fix working directory so relative paths work in both Colab and locally
import os
from pathlib import Path

notebook_dir = Path('part1_detection')
if Path('/content').exists():  # we're in Colab
    os.chdir('/content/AcneDetection/part1_detection')
else:
    # Locally: run from repo root, adjust to notebook dir
    if Path('part1_detection').exists():
        os.chdir('part1_detection')

print(f'Working directory: {os.getcwd()}')

In [ ]:
import json
import random
from pathlib import Path

import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageDraw
from torchvision import transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from ultralytics import YOLO

%matplotlib inline

In [ ]:
DATA_DIR    = Path("../data/acne04")
YOLO_CKPT   = Path("../outputs/yolov8/acne04/weights/best.pt")
FRCNN_CKPT  = Path("../outputs/faster_rcnn/best.pth")
OUT_DIR     = Path("../outputs/figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 5
CONF        = 0.25
N_IMAGES    = 6
SEED        = 42

CLASS_COLORS = {
    "papules":                 "#00FFCE",
    "whitehead and blackhead": "#C7FC00",
    "pustules":                "#FE0056",
    "nodules and cysts":       "#8622FF",
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Load models

In [ ]:
# YOLOv8
yolo_model = YOLO(str(YOLO_CKPT))
print("YOLOv8 loaded.")

# Faster R-CNN
frcnn = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
frcnn.roi_heads.box_predictor = FastRCNNPredictor(
    frcnn.roi_heads.box_predictor.cls_score.in_features, NUM_CLASSES)
frcnn.load_state_dict(torch.load(str(FRCNN_CKPT), map_location=device))
frcnn.to(device).eval()

with open(DATA_DIR / "test" / "_annotations.coco.json") as f:
    coco = json.load(f)
cats = sorted(coco["categories"], key=lambda c: c["id"])
label_map  = {i+1: c["name"] for i, c in enumerate(cats)}
id_to_name = {c["id"]: c["name"] for c in cats}
print("Faster R-CNN loaded.")

## 2. Helper: draw bounding boxes

In [ ]:
def draw_boxes(img, boxes, labels, scores=None, fallback_color="#FF0000"):
    img  = img.copy()
    draw = ImageDraw.Draw(img)
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = [int(v) for v in box]
        label = labels[i] if i < len(labels) else ""
        color = CLASS_COLORS.get(label, fallback_color)
        draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
        text  = label[:3].upper()
        if scores: text += f" {scores[i]:.2f}"
        draw.text((x1+2, max(0, y1-12)), text, fill=color)
    return img

## 3. Run inference and build comparison grid

In [ ]:
random.seed(SEED)
ann_map = {}
for ann in coco["annotations"]:
    ann_map.setdefault(ann["image_id"], []).append(ann)

annotated = [m for m in coco["images"] if m["id"] in ann_map]
sample    = random.sample(annotated, min(N_IMAGES, len(annotated)))

fig, axes = plt.subplots(N_IMAGES, 3, figsize=(15, 5 * N_IMAGES))
for col, title in enumerate(["Ground Truth", "YOLOv8s", "Faster R-CNN"]):
    axes[0][col].set_title(title, fontsize=13, fontweight="bold")

transform = transforms.ToTensor()

for row, meta in enumerate(sample):
    img_path = DATA_DIR / "test" / meta["file_name"]
    img      = Image.open(img_path).convert("RGB")

    # Ground truth
    anns      = ann_map[meta["id"]]
    gt_boxes  = [[a["bbox"][0], a["bbox"][1], a["bbox"][0]+a["bbox"][2], a["bbox"][1]+a["bbox"][3]] for a in anns]
    gt_labels = [id_to_name[a["category_id"]] for a in anns]
    img_gt    = draw_boxes(img, gt_boxes, gt_labels)

    # YOLOv8
    res        = yolo_model.predict(str(img_path), conf=CONF, verbose=False)[0]
    y_boxes    = res.boxes.xyxy.tolist()
    y_labels   = [res.names[int(c)] for c in res.boxes.cls.tolist()]
    y_scores   = res.boxes.conf.tolist()
    img_yolo   = draw_boxes(img, y_boxes, y_labels, y_scores)

    # Faster R-CNN
    with torch.no_grad():
        out = frcnn(transform(img).unsqueeze(0).to(device))[0]
    f_boxes = [b.tolist() for b, s in zip(out["boxes"], out["scores"]) if s >= CONF]
    f_labels = [label_map.get(l.item(), str(l.item())) for l, s in zip(out["labels"], out["scores"]) if s >= CONF]
    f_scores = [s.item() for s in out["scores"] if s >= CONF]
    img_frcnn = draw_boxes(img, f_boxes, f_labels, f_scores)

    for col, vis in enumerate([img_gt, img_yolo, img_frcnn]):
        axes[row][col].imshow(vis)
        axes[row][col].axis("off")

patches = [mpatches.Patch(color=c, label=l) for l, c in CLASS_COLORS.items()]
fig.legend(handles=patches, loc="lower center", ncol=2, fontsize=9, bbox_to_anchor=(0.5, 0))
plt.suptitle("ACNE04 Test Set — YOLOv8 vs Faster R-CNN", fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / "detection_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/figures/detection_comparison.png")